# Tarea 2

## Cuerpo docente

- Profesores: [Andrés Abeliuk](https://aabeliuk.github.io/), Fabrizzio Pezzolla
- Profesor auxiliar: Rodrigo Molina

## Instrucciones generales

- Grupos de máximo 4 personas.
- Está prohibido compartir las respuestas con otros grupos.
- Indicios de copia serán penalizados con la nota mínima.
- Cualquier duda fuera del horario de clases, al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente; si utilizan material extra, deben citarlo.


## Integrantes

> POR FAVOR AGREGAR TODOS LOS NOMBRES DE LOS INTEGRANTES

# 0. Preparación

Esta tarea necesita una GPU. En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

In [ ]:
%%capture

!pip install gensim

In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from gensim.models import Word2Vec
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
)

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
dispositivo

Carguen el dataset de la Tarea 1 y reemplacen `SEMILLA` por el número de su grupo.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/dccuchile/CC6205/master/assignments/new/assignment_1/train/train.tsv", sep="\t")

SEMILLA = 0  # TODO: reemplacen por el número de su grupo

Usamos la misma separación en entrenamiento y prueba de la Tarea 1: 9.544 textos de entrenamiento y 2.386 de prueba, cada uno etiquetado como `odio`, `incivilidad` o `normal`. Recordemos las definiciones:

- Se considera *discurso de odio* a cualquier expresión que promueva o incite a la discriminación, la hostilidad o la violencia hacia una persona o grupo de personas en una relación asimétrica de poder, tales como la raza, la etnia, el género, la orientación sexual, la religión, la nacionalidad, una discapacidad u otra característica similar.
- Se considera *incivilidad* a cualquier comportamiento o actitud que rompe las normas de respeto, cortesía y consideración en la interacción entre personas. Esta puede manifestarse de diversas formas, tales como insultos, ataques personales, sarcasmo, desprecio, entre otras.
- Los textos que no corresponden ni a `odio` ni a `incivilidad` son `normal`.

In [ ]:
# Sin textos con etiquetas en conflicto ni duplicados
etiquetas_por_texto = df.groupby("texto")["clase"].nunique()
en_conflicto = etiquetas_por_texto[etiquetas_por_texto > 1].index
df_limpio = df[~df["texto"].isin(en_conflicto)].drop_duplicates("texto")

# La misma separación para todos los grupos
X_train, X_test, y_train, y_test = train_test_split(
    df_limpio["texto"], df_limpio["clase"],
    test_size=0.2, stratify=df_limpio["clase"], random_state=0,
)

clases = sorted(y_train.unique())
y_test.value_counts()

La función `evaluar` es la de la Tarea 1, completa, salvo que recibe las etiquetas predichas en vez del clasificador. Así sirve para cualquier modelo. Guardaremos las predicciones de cada modelo en el diccionario `predicciones`.

In [ ]:
def evaluar(
    y: pd.Series, prediccion: pd.Series, mostrar: bool = True
) -> pd.Series:
    """Muestra la matriz de confusión y calcula recall y exactitud.

    Devuelve una Series con el recall de cada clase, seguido de la
    exactitud.
    """
    matriz = pd.crosstab(
        y, prediccion, rownames=["real"], colnames=["predicción"]
    )
    # Las mismas clases en filas y columnas
    matriz = matriz.reindex(index=clases, columns=clases, fill_value=0)
    if mostrar:
        display(matriz)

    recall = pd.Series(np.diag(matriz) / matriz.sum(axis=1), index=clases)
    exactitud = np.trace(matriz) / matriz.to_numpy().sum()
    return pd.concat([recall, pd.Series({"exactitud": exactitud})])


predicciones = {}

El punto de comparación es el clasificador TF-IDF de la Tarea 1, con las opciones por defecto de `TfidfVectorizer`.

In [ ]:
# Línea base: TF-IDF y regresión logística
vectorizador = TfidfVectorizer(min_df=2)
tfidf_clf = LogisticRegression(max_iter=2000)
tfidf_clf.fit(vectorizador.fit_transform(X_train), y_train)

predicciones["TF-IDF"] = pd.Series(
    tfidf_clf.predict(vectorizador.transform(X_test)), index=X_test.index
)
evaluar(y_test, predicciones["TF-IDF"]).round(3)

# 1. Red recurrente

En la Tarea 1 representaron cada texto como una bolsa de palabras (TF-IDF, LSA) o como el promedio de sus vectores Word2Vec. En ambos casos el orden de las palabras se pierde. Una red recurrente, en cambio, lee el texto palabra por palabra y lo resume en un estado oculto (clase 4). En esta sección veremos si eso basta para superar a TF-IDF en un corpus de unos 10.000 tweets.

Las siguientes funciones son las del notebook de la clase 4:

- `tokenizar`, `crear_vocabulario` y `lote` convierten los textos en secuencias de ids con relleno.
- `Clasificador` es una capa de embedding, seguida de una GRU (o una LSTM) y de una capa lineal sobre el último estado oculto.
- `entrenar` y `predecir` lo entrenan con Adam y lo usan para clasificar.

In [ ]:
def tokenizar(texto: str) -> list[str]:
    """Devuelve las palabras en minúsculas."""
    return re.findall(r"\w+", texto.lower())


def crear_vocabulario(
    listas: list[list[str]], minimo: int = 2
) -> dict[str, int]:
    """Asigna un id a cada palabra que aparece al menos `minimo` veces.

    El id 0 es el relleno y el 1, las palabras desconocidas.
    """
    frecuencias = Counter(p for tokens in listas for p in tokens)
    vocab = {"<pad>": 0, "<unk>": 1}
    for palabra, n in frecuencias.most_common():
        if n >= minimo:
            vocab[palabra] = len(vocab)
    return vocab


def lote(
    textos: pd.Series, etiquetas: pd.Series, vocab: dict[str, int]
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Devuelve los ids con relleno, los largos y las etiquetas."""
    ids = [
        torch.tensor([vocab.get(p, 1) for p in tokenizar(t)] or [1])
        for t in textos
    ]
    X = nn.utils.rnn.pad_sequence(ids, batch_first=True)
    largos = torch.tensor([len(i) for i in ids])
    y = torch.tensor([clases.index(c) for c in etiquetas])
    return X, largos, y

In [ ]:
class Clasificador(nn.Module):
    def __init__(self, n_palabras, celda=nn.GRU, d=64, h=64, salidas=3):
        super().__init__()
        self.embedding = nn.Embedding(n_palabras, d, padding_idx=0)
        self.recurrente = celda(d, h, batch_first=True)
        self.salida = nn.Linear(h, salidas)

    def forward(self, X, largos):
        E = self.embedding(X)
        E = nn.utils.rnn.pack_padded_sequence(
            E, largos, batch_first=True, enforce_sorted=False
        )
        _, h = self.recurrente(E)
        if isinstance(h, tuple):   # la LSTM devuelve (estado, memoria)
            h = h[0]
        return self.salida(h[-1])


def entrenar(
    modelo: nn.Module,
    datos: tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    epocas: int = 4,
) -> nn.Module:
    """Entrena el modelo con Adam y entropía cruzada."""
    modelo.to(dispositivo)
    X, largos, y = datos
    X, y = X.to(dispositivo), y.to(dispositivo)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=2e-3)
    perdida = nn.CrossEntropyLoss()
    for epoca in range(epocas):
        modelo.train()
        for i in torch.randperm(len(X)).split(64):
            optimizador.zero_grad()
            perdida(modelo(X[i], largos[i]), y[i]).backward()
            optimizador.step()
    return modelo.eval()


def predecir(
    modelo: nn.Module, datos: tuple[torch.Tensor, torch.Tensor, torch.Tensor]
) -> list[str]:
    """Devuelve la clase predicha para cada texto."""
    X, largos, _ = datos
    with torch.no_grad():
        ids = modelo(X.to(dispositivo), largos).argmax(1)
    return [clases[i] for i in ids.tolist()]

## 1.1 Clasificador GRU

Entrenen un `Clasificador` con los textos de entrenamiento, durante 4 épocas y con los demás valores por defecto, y evalúenlo con los textos de prueba. Usen `torch.manual_seed(SEMILLA)` antes de crear el modelo. **(0,2 pts)**

In [ ]:
# TODO 1: GRU
vocab = ...
gru = ...

predicciones["GRU"] = ...  # una Series con el índice de X_test
evaluar(y_test, predicciones["GRU"]).round(3)

---

**📝 Pregunta 1 (0,4 pts):** Comparen la GRU con TF-IDF clase por clase: ¿en qué clases gana o pierde cada uno? Entrenen la GRU con otras dos semillas: ¿las diferencias con TF-IDF son mayores que la variación entre semillas? ¿Por qué una red que considera el orden de las palabras no supera claramente a una bolsa de palabras en este corpus?
> **Respuesta:**

---

## 1.2 Vectores de palabras preentrenados

Entrenen Word2Vec con los textos de entrenamiento, con los parámetros de abajo, y usen sus vectores como valores iniciales de la capa `embedding` de un nuevo `Clasificador`. Las palabras del vocabulario que no estén en Word2Vec conservan su valor al azar. Entrénenlo y evalúenlo como en la sección 1.1. **(0,3 pts)**

---

**📝 Pregunta 2 (0,4 pts):**

**(a) (0,2 pts)** Respondan **antes** de entrenar el modelo: ¿qué debería cambiar al partir de los vectores de Word2Vec en vez de vectores al azar? ¿En qué clase esperan el mayor efecto y por qué?

**(b) (0,2 pts)** Después de entrenarlo: ¿se cumplió su hipótesis? Compárenlo con la GRU de la sección 1.1 en exactitud y en el recall de cada clase.

> **Respuesta (a):**

> **Respuesta (b):**

---

In [ ]:
# workers=1 para que el resultado sea reproducible
parametros = dict(sg=1, vector_size=64, window=5, min_count=2, negative=10,
                  epochs=5, seed=0, workers=1)

# TODO 2: Word2Vec y GRU con esos vectores iniciales
w2v = ...
gru_w2v = ...

predicciones["GRU + Word2Vec"] = ...
evaluar(y_test, predicciones["GRU + Word2Vec"]).round(3)

# 2. Transformer preentrenado

En la clase 5 vimos que un transformer preentrenado entrega un vector por cada token, y que ese vector depende del contexto en que aparece el token. En esta sección lo usaremos **congelado**, sin ajustar sus pesos, como un vectorizador más: cada texto se convierte en un vector, y sobre esos vectores se entrena la misma regresión logística de la Tarea 1.

Usaremos [RoBERTuito](https://huggingface.co/pysentimiento/robertuito-base-uncased), un modelo del tipo RoBERTa preentrenado con tweets en español.

In [ ]:
MODELO = "pysentimiento/robertuito-base-uncased"

tokenizador = AutoTokenizer.from_pretrained(MODELO)
transformer = AutoModel.from_pretrained(MODELO).to(dispositivo).eval()

## 2.1 Vectores de documento

Completen `vectores`, que pasa los textos por el transformer en lotes. Cada texto queda representado por el promedio de los vectores de sus tokens en la última capa, sin contar el relleno (padding). **(0,3 pts)**

In [ ]:
def vectores(textos: list[str], tamano_lote: int = 64) -> np.ndarray:
    """Devuelve un vector por texto: el promedio de sus tokens.

    Los vectores son los de la última capa del transformer.
    """
    partes = []
    for i in range(0, len(textos), tamano_lote):
        entrada = tokenizador(
            textos[i:i + tamano_lote], padding=True, truncation=True,
            max_length=128, return_tensors="pt",
        ).to(dispositivo)
        with torch.no_grad():
            H = transformer(**entrada).last_hidden_state
        mascara = entrada["attention_mask"]

        # TODO 3: promedio de los vectores de los tokens, sin relleno
        promedio = ...

        partes.append(promedio.cpu().numpy())
    return np.concatenate(partes)

La siguiente celda calcula los vectores de todos los textos.

In [ ]:
trf_train = vectores(X_train.tolist())
trf_test = vectores(X_test.tolist())
trf_train.shape

Entrenen una regresión logística con `max_iter=2000`, como en la Tarea 1, con estos vectores, y evalúenla. **(0,1 pts)**

In [ ]:
# TODO 4: regresión logística sobre los vectores del transformer
trf_clf = ...

predicciones["Transformer"] = ...
evaluar(y_test, predicciones["Transformer"]).round(3)

---

**📝 Pregunta 3 (0,4 pts):** Comparen el transformer con TF-IDF y con las dos GRU, en exactitud y en el recall de cada clase. ¿Qué tiene este modelo que no tenían las GRU? Consideren con qué textos, y con cuántos, aprendió cada modelo sus representaciones.
> **Respuesta:**

---

## 2.2 Polisemia

En la Pregunta 11 de la Tarea 1 vieron que, en Word2Vec, una palabra con más de un sentido tiene un solo vector. Aquí usaremos dos palabras del corpus con varios sentidos: `pobre` (sin dinero, o digno de compasión, como en “pobre mamá”) y `rojo` (el color, o alguien de izquierda, entre otros). La siguiente función devuelve el vector de una palabra dentro de un texto: el promedio de los vectores de los tokens en que el transformer la divide.

In [ ]:
def vector_en_contexto(texto: str, palabra: str) -> np.ndarray:
    """Devuelve el vector de `palabra` en su primera aparición.

    Promedia los vectores de la última capa de los tokens que forman
    la palabra.
    """
    inicio = re.search(rf"\b{palabra}\b", texto, re.IGNORECASE).start()
    fin = inicio + len(palabra)
    entrada = tokenizador(
        texto, truncation=True, max_length=128,
        return_offsets_mapping=True, return_tensors="pt",
    )
    posiciones = entrada.pop("offset_mapping")[0]
    en_palabra = (posiciones[:, 0] < fin) & (posiciones[:, 1] > inicio)
    with torch.no_grad():
        H = transformer(**entrada.to(dispositivo)).last_hidden_state[0]
    return H[en_palabra.to(dispositivo)].mean(0).cpu().numpy()

Para cada palabra, elijan seis textos de entrenamiento en que aparezca, con al menos dos sentidos distintos. Calculen el vector de la palabra en cada texto y muestren la matriz de similitud coseno entre las seis apariciones, junto con la clase de cada texto. **(0,2 pts)**

In [ ]:
# TODO 5: similitud entre apariciones de pobre y de rojo

---

**📝 Pregunta 4 (0,4 pts):** ¿Las apariciones de una palabra con el mismo sentido quedan más cerca entre sí que las de sentidos distintos? ¿Hay excepciones? Comparen con su respuesta a la Pregunta 11 (b) de la Tarea 1: ¿qué cambia al tener un vector por aparición, en vez de un vector por palabra?
> **Respuesta:**

---

# 3. Modelo de lenguaje con instrucciones

En la clase 6 vimos que un LLM ajustado con instrucciones puede resolver una tarea nueva sin entrenamiento, si se la describimos en el prompt: solo con la instrucción (*zero-shot*) o agregando algunos ejemplos resueltos (*few-shot*).

Usaremos [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507), un modelo abierto de 4.000 millones de parámetros. En `float16` ocupa unos 10 GB de memoria de GPU, así que cabe en la T4 de Colab. Clasificar los textos de esta sección toma algunos minutos.

`a_etiqueta` busca la primera clase que aparece en la respuesta, y `responder` genera como máximo 5 tokens. Por eso su prompt debe pedirle al modelo que responda **solo** con una de las tres etiquetas: `odio`, `incivilidad` o `normal`.

In [ ]:
MODELO_LLM = "Qwen/Qwen3-4B-Instruct-2507"

tokenizador_llm = AutoTokenizer.from_pretrained(
    MODELO_LLM, padding_side="left"
)
llm = AutoModelForCausalLM.from_pretrained(
    MODELO_LLM, torch_dtype=torch.float16
).to(dispositivo).eval()

La función `responder` recibe una lista de conversaciones, cada una una lista de mensajes con el formato de chat de Hugging Face, y devuelve la respuesta del modelo a cada una. `a_etiqueta` convierte una respuesta en una de las tres clases.

In [ ]:
def responder(
    conversaciones: list[list[dict[str, str]]],
    max_tokens: int = 5,
    tamano_lote: int = 8,
) -> list[str]:
    """Devuelve la respuesta del modelo a cada conversación."""
    respuestas = []
    for i in range(0, len(conversaciones), tamano_lote):
        entrada = tokenizador_llm.apply_chat_template(
            conversaciones[i:i + tamano_lote], add_generation_prompt=True,
            padding=True, return_tensors="pt", return_dict=True,
        ).to(dispositivo)
        with torch.no_grad():
            salida = llm.generate(
                **entrada, max_new_tokens=max_tokens, do_sample=False
            )
        nuevos = salida[:, entrada["input_ids"].shape[1]:]
        respuestas += tokenizador_llm.batch_decode(
            nuevos, skip_special_tokens=True
        )
    return respuestas


def a_etiqueta(respuesta: str) -> str:
    """Devuelve la primera clase mencionada, o `normal` si no hay."""
    for palabra in re.findall(r"\w+", respuesta.lower()):
        if palabra in clases:
            return palabra
    return "normal"

Los modelos de esta sección son lentos, así que usaremos 500 textos de prueba, los mismos para todos los grupos.

In [ ]:
X_llm, _, y_llm, _ = train_test_split(
    X_test, y_test, train_size=500, stratify=y_test, random_state=0
)
y_llm.value_counts()

## 3.1 Zero-shot

Completen `conversacion_zero_shot`, que recibe un texto y devuelve la conversación que se le entrega al modelo: una lista de mensajes `{"role": ..., "content": ...}`. Pueden poner las instrucciones en un mensaje de sistema (`"system"`) y el texto a clasificar en un mensaje de usuario (`"user"`). El prompt debe incluir las definiciones de odio e incivilidad de la Tarea 1. Luego clasifiquen los 500 textos de prueba. **(0,3 pts)**

In [ ]:
# TODO 6: prompt zero-shot
def conversacion_zero_shot(texto: str) -> list[dict[str, str]]:
    ...


respuestas = responder([conversacion_zero_shot(t) for t in X_llm])
predicciones["LLM zero-shot"] = pd.Series(
    [a_etiqueta(r) for r in respuestas], index=X_llm.index
)
evaluar(y_llm, predicciones["LLM zero-shot"]).round(3)

En la sección 1.2 de la Tarea 1 etiquetaron a mano cinco textos. Clasifíquenlos con su prompt zero-shot. **(0,1 pts)**

In [ ]:
ids_ejercicio = [475, 3504, 10590, 1449, 3653]
ejercicio = df.set_index("id").loc[ids_ejercicio, "texto"]

# TODO 7: los cinco textos con el prompt zero-shot

---

**📝 Pregunta 5 (0,4 pts):** Comparen las etiquetas que el modelo les dio a los cinco textos con las que ustedes les asignaron en la Pregunta 2 de la Tarea 1 y con las del dataset. ¿El modelo coincide más con ustedes o con el dataset? Cuando difiere, ¿se equivocó el modelo o la etiqueta es discutible?
> **Respuesta:**

---

## 3.2 Few-shot

Agreguen ejemplos al prompt: k textos de entrenamiento por clase, con su etiqueta, elegidos al azar con `random_state=SEMILLA` (por ejemplo, con `groupby(...).sample(k, random_state=SEMILLA)`). Pueden incluirlos como pares de mensajes `"user"` y `"assistant"` antes del texto a clasificar, o dentro del mensaje de sistema. Clasifiquen los 500 textos de prueba con k = 1 y con k = 3, y guarden las predicciones en `predicciones["LLM few-shot k=1"]` y `predicciones["LLM few-shot k=3"]`. **(0,3 pts)**

In [ ]:
# TODO 8: prompt few-shot con k ejemplos por clase

---

**📝 Pregunta 6 (0,4 pts):** Comparen zero-shot y few-shot en exactitud y en el recall de cada clase. ¿Qué cambia con los ejemplos, y en qué clases? ¿Mejora el modelo al pasar de k = 1 a k = 3? Con 500 textos de prueba, ¿qué diferencia de exactitud considerarían ruido?
> **Respuesta:**

---

# 4. Comparación

## 4.1 Todos los modelos

La siguiente celda junta el recall y la exactitud de todos los modelos en los 500 textos de la sección 3.

In [ ]:
pd.DataFrame({
    nombre: evaluar(y_llm, p[X_llm.index], mostrar=False)
    for nombre, p in predicciones.items()
}).round(3)

## 4.2 Curva de aprendizaje

¿Cuántos textos etiquetados hacen falta? Entrenen de nuevo TF-IDF, la GRU de la sección 1.1 y la regresión logística sobre los vectores del transformer con 250, 1.000 y 4.000 textos de entrenamiento, elegidos con `train_test_split(..., train_size=n, stratify=y_train, random_state=SEMILLA)`, y con todos. Grafiquen la exactitud en el conjunto de prueba según el número de textos, con el eje horizontal en escala logarítmica, y agreguen los modelos de la sección 3 como líneas horizontales, con su exactitud en los 500 textos. **(0,4 pts)**

In [ ]:
# TODO 9: curva de aprendizaje
tamanos = [250, 1000, 4000, len(X_train)]

---

**📝 Pregunta 7 (0,5 pts):** Según su gráfico, ¿desde cuántos textos etiquetados supera cada modelo al LLM? ¿Qué modelo conviene cuando hay pocos datos etiquetados y cuál cuando hay muchos? ¿Por qué el transformer congelado necesita menos textos que la GRU? Revisen también el recall de `odio` en la tabla de la sección 4.1: ¿la exactitud cuenta toda la historia?
> **Respuesta:**

---

# 5. Preguntas finales

---

**📝 Pregunta 8 (0,4 pts):** Vuelvan a la Pregunta 10 de la Tarea 1: una red social quiere usar su modelo para moderar. Con el mejor modelo de las secciones 1 y 2, en el conjunto de prueba completo, ¿cuántos textos de `odio` se clasifican como `normal`, y cuántos `normal` como `odio`? Compárenlo con TF-IDF. ¿Qué modelo usarían para moderar, y por qué?
> **Respuesta:**

---

---

**📝 Pregunta 9 (0,5 pts):** Comparen la exactitud de cada solución de esta tarea (TF-IDF, GRU, GRU + Word2Vec, transformer congelado, LLM zero-shot y few-shot) con su costo: cuántos datos etiquetados necesita, cuánto tarda en entrenarse y en predecir, y qué hardware requiere (CPU o GPU, memoria). Si tuvieran que clasificar un millón de tweets al día, ¿cuál usarían? ¿Y si tuvieran que empezar mañana, sin datos etiquetados?
> **Respuesta:**

---